# Module 4 — Flooding / Waterlogging (maize, GHA)
The **wet-side** hazard WRSI can't see. Two metrics: **SPI-3 wet anomaly** (validated, surface/seasonal excess) and the **AquaCrop aeration-stress** soil-water model (root-zone water above field capacity, from SoilGrids/Saxton — modelled, uncalibrated).

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

In [ ]:
!pip -q install earthengine-api folium pandas geopandas 2>/dev/null
print('installed.')

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

In [ ]:
# --- config + robust folium map (EE tile URLs; no geemap) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import folium
def ee_layer(fmap,image,vis,name):
    m=ee.Image(image).getMapId(vis)
    folium.TileLayer(m["tile_fetcher"].url_format,attr="Google Earth Engine",name=name,overlay=True,control=True).add_to(fmap)
def new_map(zoom=7):
    c=aoi_run.centroid(1).coordinates().getInfo(); return folium.Map(location=[c[1],c[0]],zoom_start=zoom,tiles="CartoDB positron")
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting (onset anchor)

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

In [ ]:
# --- excess / waterlogging ---
from src import excess as EX, soil as SOIL
mz=kc['maize']; d_veg=mz['L_ini']+mz['L_dev']; d_flo=d_veg+mz['L_mid']; lgp=mz['LGP_dekads']
wet=EX.spi3_wet(ee,aoi_run,YEAR,end_month=5 if SEASON=='Long rains' else (9 if SEASON=='Meher' else 12))
hy=SOIL.build_hydro_mm(ee,root_depth_cm=100)   # FC/SAT/tau from SoilGrids + Saxton-Rawls
wl=EX.aeration_stress_index(ee,aoi_run,planting,YEAR,d_veg,d_flo,lgp,ss,se,hy['FC_mm'],hy['SAT_mm'],hy['tau'])
print('excess/waterlogging computed')

In [ ]:
M=new_map()
ee_layer(M, wet.updateMask(mask).clip(aoi_run), {'min':0,'max':1,'palette':['ffffff','3690c0']}, 'SPI-3 very wet (excess, validated)')
ee_layer(M, wl.updateMask(mask).clip(aoi_run), {'min':0,'max':40,'palette':['f7fbff','6baed6','08306b']}, 'Soil waterlogging (modelled, uncal.)')
folium.LayerControl().add_to(M); M

*SPI-3 wet = **surface/seasonal** anomaly (validated); aeration = **root-zone** soil saturation (modelled, needs calibration) — two different hazards. See `WATERLOGGING_METHODOLOGY`.*